# Model Building

# Importing Libraries

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report,accuracy_score,recall_score,r2_score,precision_score,roc_auc_score,confusion_matrix,\
f1_score,cohen_kappa_score

from imblearn.over_sampling import SMOTE

import warnings 
warnings.filterwarnings("ignore")

## Splitting the dataset

In [13]:
PROJECT_ROOT = Path.cwd().parent

train_df = pd.read_csv(PROJECT_ROOT / "data" / "train_sample_2024.csv")
test_df  = pd.read_csv(PROJECT_ROOT / "data" / "test_sample_2025.csv")

In [14]:
print(train_df.shape)
print(test_df.shape)

(200001, 25)
(100001, 25)


## Data Preprocessing


In [15]:
# Extract hour
train_df['DEP_HOUR'] = train_df['CRS_DEP_TIME'] // 100
test_df['DEP_HOUR'] = test_df['CRS_DEP_TIME'] // 100

# Create time-of-day category
def time_period(hour):
    if 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

train_df['DEP_PERIOD'] = train_df['DEP_HOUR'].apply(time_period)
test_df['DEP_PERIOD'] = test_df['DEP_HOUR'].apply(time_period)

In [16]:
# Applying N-1 encoding
train_cat = ['DEP_PERIOD', 'MKT_UNIQUE_CARRIER']
train_df = pd.get_dummies(train_df, columns=train_cat, drop_first=True)
test_df = pd.get_dummies(test_df, columns=train_cat, drop_first=True)

In [19]:
feature_cols = [
    'QUARTER',
    'MONTH',
    'DAY_OF_WEEK',
    'DEP_HOUR',
    'ORIGIN_AIRPORT_ID',
    'DEST_AIRPORT_ID',
    'DISTANCE',
    'TAXI_OUT'
]
dep_period_dummies = [col for col in train_df.columns if col.startswith('DEP_PERIOD_')]
carrier_dummies = [col for col in train_df.columns if col.startswith('MKT_UNIQUE_CARRIER_')]

feature_cols = feature_cols + dep_time_dummies + carrier_dummies


test_num=test_df[feature_cols].select_dtypes(include=np.number).columns.to_list()
test_cat=test_df[feature_cols].select_dtypes(include=object).columns.to_list()

In [22]:
train_df.columns

Index(['YEAR', 'QUARTER', 'MONTH', 'DAY_OF_WEEK', 'ORIGIN_AIRPORT_ID',
       'ORIGIN', 'DEST_AIRPORT_ID', 'DEST', 'CRS_DEP_TIME', 'DEP_TIME_BLK',
       'TAXI_OUT', 'TAXI_IN', 'CANCELLED', 'DIVERTED', 'ACTUAL_ELAPSED_TIME',
       'AIR_TIME', 'DISTANCE', 'DISTANCE_GROUP', 'CARRIER_DELAY',
       'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY',
       'OPERATIONAL_RISK', 'DEP_HOUR', 'DEP_PERIOD_Evening',
       'DEP_PERIOD_Morning', 'DEP_PERIOD_Night', 'MKT_UNIQUE_CARRIER_AS',
       'MKT_UNIQUE_CARRIER_B6', 'MKT_UNIQUE_CARRIER_DL',
       'MKT_UNIQUE_CARRIER_F9', 'MKT_UNIQUE_CARRIER_G4',
       'MKT_UNIQUE_CARRIER_HA', 'MKT_UNIQUE_CARRIER_NK',
       'MKT_UNIQUE_CARRIER_UA', 'MKT_UNIQUE_CARRIER_WN'],
      dtype='str')

In [91]:
# Create x and y for train
x_train = train_df[feature_cols]
y_train = train_df['OPERATIONAL_RISK']

# Create x and y for test
x_test = test_df[feature_cols]
y_test=test_df['OPERATIONAL_RISK']

In [92]:
# # Appling SMOTE to balance the class
# smote = SMOTE(random_state=42)

# x_train_smote, y_train_smote = smote.fit_resample(x_train, y_train)

# print("Before SMOTE:\n", y_train.value_counts())
# print("After SMOTE:\n", y_train_smote.value_counts())

In [93]:
summary = pd.DataFrame(columns=['Name','Accuracy','Precision','Recall','F1_Score','ROC_AUC','Cohen_Kappa'])

def metrics(name, y_test, ypred_hard, ypred_soft):
    global summary

    print(f"\nClassification Report for {name}:\n")
    print(classification_report(y_test, ypred_hard))
    print('------------------------------------------------------------')

    acc = round(accuracy_score(y_test, ypred_hard), 2)
    pre = round(precision_score(y_test, ypred_hard), 2)
    rec = round(recall_score(y_test, ypred_hard), 2)
    f1  = round(f1_score(y_test, ypred_hard), 2)
    auc = round(roc_auc_score(y_test, ypred_soft), 2)
    reliability = round(cohen_kappa_score(y_test, ypred_hard), 2)

    result = pd.DataFrame({
        'Name':[name],
        'Accuracy':[acc],
        'Precision':[pre],
        'Recall':[rec],
        'F1_Score':[f1],
        'ROC_AUC':[auc],
        'Cohen_Kappa':[reliability]
    })

    summary = pd.concat([summary, result], ignore_index=True)
    return summary

## Baseline Model - Logistic Regression

In [ ]:
model1 = LogisticRegression(random_state=42)
model1.fit(x_train, y_train)

ypred_soft_1 = model1.predict_proba(x_test)[:,1]
ypred_hard_1 = (ypred_soft_1 > 0.5).astype(int)

summary = metrics(
    "Baseline Logistic",
    y_test,
    ypred_hard_1,
    ypred_soft_1
)


Classification Report for Baseline Logistic:

              precision    recall  f1-score   support

           0       0.86      0.64      0.74     78229
           1       0.33      0.64      0.43     21772

    accuracy                           0.64    100001
   macro avg       0.60      0.64      0.58    100001
weighted avg       0.75      0.64      0.67    100001

------------------------------------------------------------


In [95]:
summary

,Name,Accuracy,Precision,Recall,F1_Score,ROC_AUC,Cohen_Kappa
0,Baseline Logistic,0.64,0.33,0.64,0.43,0.7,0.21


## Model 2 - Logistic Regression with class weights

In [96]:
model2 = LogisticRegression(class_weight='balanced', random_state=42)
model2.fit(x_train, y_train)

ypred_soft_2 = model2.predict_proba(x_test)[:,1]
ypred_hard_2 = (ypred_soft_2 > 0.5).astype(int)

summary = metrics(
    "Logistic class",
    y_test,
    ypred_hard_2,
    ypred_soft_2
)


Classification Report for Logistic class:

              precision    recall  f1-score   support

           0       0.87      0.65      0.74     78229
           1       0.34      0.65      0.44     21772

    accuracy                           0.65    100001
   macro avg       0.60      0.65      0.59    100001
weighted avg       0.75      0.65      0.68    100001

------------------------------------------------------------


In [97]:
summary

,Name,Accuracy,Precision,Recall,F1_Score,ROC_AUC,Cohen_Kappa
0,Baseline Logistic,0.64,0.33,0.64,0.43,0.7,0.21
1,Logistic class,0.65,0.34,0.65,0.44,0.71,0.22


In [98]:
# model3 = DecisionTreeClassifier(class_weight='balanced',random_state=42,max_depth=100)
# model3.fit(x_train, y_train)

# ypred_soft_3 = model3.predict_proba(x_test)[:,1]
# ypred_hard_3 = (ypred_soft_3 > 0.3).astype(int)

# summary = metrics(
#     "Decision Tree",
#     y_test,
#     ypred_hard_3,
#     ypred_soft_3
# )

In [99]:
# summary

In [100]:
# model4 = RandomForestClassifier(class_weight='balanced',random_state=42,max_depth=100)
# model4.fit(x_train, y_train)

# ypred_soft_4 = model4.predict_proba(x_test)[:,1]
# ypred_hard_4 = (ypred_soft_4 > 0.5).astype(int)

# summary = metrics(
#     "Random Forest",
#     y_test,
#     ypred_hard_4,
#     ypred_soft_4
# )

In [101]:
# summary

In [102]:
# model5 = XGBClassifier(
#     n_estimators=10000)

# model5.fit(x_train, y_train)
# ypred_soft_5 = model5.predict_proba(x_test)[:,1]
# ypred_hard_5 = (ypred_soft_5 > 0.5).astype(int)
# summary = metrics(
#     "XGBoost",
#     y_test,
#     ypred_hard_5,
#     ypred_soft_5
# )
